# LAB 3 — Auto Loader Schema Evolution

This notebook adds evolved and renamed source files to the Auto Loader landing directory, observes how Auto Loader reacts, and performs a separate rescued-data test.

## 1. Load shared configuration

In [0]:
%run ./lab03_config

## 2. Define helper functions

In [0]:
def copy_json_files(
    source_directory: str,
    target_directory: str,
    prefix: str
) -> int:
    copied_files = 0

    for file_info in dbutils.fs.ls(source_directory):
        if not file_info.name.endswith(".json"):
            continue

        destination = (
            f"{target_directory}/{prefix}_{file_info.name}"
        )

        dbutils.fs.cp(
            file_info.path,
            destination
        )
        copied_files += 1

    return copied_files


def count_json_files(path: str) -> int:
    return len(
        [
            file_info
            for file_info in dbutils.fs.ls(path)
            if file_info.name.endswith(".json")
        ]
    )


def run_main_autoloader(max_retries: int = 1):
    from pyspark.sql.functions import col, current_timestamp

    stream_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option(
            "cloudFiles.schemaLocation",
            autoloader_schema_path
        )
        .option(
            "cloudFiles.schemaEvolutionMode",
            "addNewColumns"
        )
        .option(
            "cloudFiles.maxFilesPerTrigger",
            max_files_per_trigger
        )
        .option(
            "rescuedDataColumn",
            rescued_data_column
        )
        .option("cloudFiles.inferColumnTypes", "true")
        .load(landing_path)
        .withColumn(
            "_source_file",
            col("_metadata.file_path")
        )
        .withColumn(
            "_source_file_name",
            col("_metadata.file_name")
        )
        .withColumn(
            "_ingested_at",
            current_timestamp()
        )
    )

    attempts = 0

    while True:
        try:
            query = (
                stream_df.writeStream
                .format("delta")
                .outputMode("append")
                .option(
                    "checkpointLocation",
                    autoloader_checkpoint_path
                )
                .option("mergeSchema", "true")
                .trigger(availableNow=True)
                .queryName("lab03_schema_evolution")
                .toTable(file_bronze_table)
            )

            query.awaitTermination()
            return query

        except Exception as exc:
            attempts += 1
            print(
                f"Auto Loader stopped during schema evolution "
                f"(attempt {attempts}): {exc}"
            )

            if attempts > max_retries:
                raise

            print(
                "Retrying with the same schema location and checkpoint. "
                "This is expected after Auto Loader records a new column."
            )

## 3. Confirm the initial Bronze table exists

In [0]:
if not spark.catalog.tableExists(file_bronze_table):
    raise RuntimeError(
        f"Run lab03_02_autoloader_initial_load first. "
        f"Missing table: {file_bronze_table}"
    )

initial_columns = spark.table(file_bronze_table).columns

print("Initial Bronze columns:")
for column_name in initial_columns:
    print(column_name)

## 4. Add files containing a new column

In [0]:
evolved_source_count = count_json_files(
    staging_evolved_path
)

if evolved_source_count == 0:
    raise FileNotFoundError(
        f"No evolved JSON files found: {staging_evolved_path}"
    )

copied_evolved_files = copy_json_files(
    staging_evolved_path,
    landing_path,
    "evolved"
)

print(f"Evolved files copied: {copied_evolved_files}")

## 5. Run Auto Loader and observe the new column

In [0]:
evolved_query = run_main_autoloader(max_retries=1)

print("Evolved-schema ingestion completed.")

In [0]:
bronze_after_evolution = spark.table(file_bronze_table)

if "source_system" not in bronze_after_evolution.columns:
    raise RuntimeError(
        "The expected source_system column was not added."
    )

display(
    bronze_after_evolution
    .filter("source_system IS NOT NULL")
    .select(
        "source_system",
        "_source_file_name",
        "_ingested_at"
    )
    .limit(20)
)

print("New column confirmed: source_system")

## 6. Add files containing a renamed field

In [0]:
renamed_source_count = count_json_files(
    staging_renamed_path
)

if renamed_source_count == 0:
    raise FileNotFoundError(
        f"No renamed JSON files found: {staging_renamed_path}"
    )

copied_renamed_files = copy_json_files(
    staging_renamed_path,
    landing_path,
    "renamed"
)

print(f"Renamed-field files copied: {copied_renamed_files}")

## 7. Run Auto Loader and inspect rename behavior

In [0]:
renamed_query = run_main_autoloader(max_retries=1)

print("Renamed-field ingestion completed.")

In [0]:
bronze_after_rename = spark.table(file_bronze_table)

expected_columns = {
    "fare_amount",
    "base_fare_amount",
}

missing_expected_columns = (
    expected_columns - set(bronze_after_rename.columns)
)

if missing_expected_columns:
    raise RuntimeError(
        f"Expected rename-test columns are missing: "
        f"{missing_expected_columns}"
    )

display(
    bronze_after_rename
    .filter("base_fare_amount IS NOT NULL")
    .select(
        "fare_amount",
        "base_fare_amount",
        "_source_file_name"
    )
    .limit(20)
)

print(
    "Auto Loader treated base_fare_amount as a new column. "
    "It did not interpret it as a business rename."
)

## 8. Demonstrate how Silver can normalize the renamed fields

In [0]:
from pyspark.sql.functions import coalesce, col

normalized_preview_df = (
    bronze_after_rename
    .withColumn(
        "normalized_fare_amount",
        coalesce(
            col("fare_amount"),
            col("base_fare_amount")
        )
    )
)

display(
    normalized_preview_df
    .select(
        "fare_amount",
        "base_fare_amount",
        "normalized_fare_amount",
        "_source_file_name"
    )
    .filter(
        "base_fare_amount IS NOT NULL"
    )
    .limit(20)
)

## 9. Run a separate rescued-data test

The malformed files are read with an explicit schema and `rescue` mode. A separate checkpoint and target table keep this experiment isolated.

In [0]:
rescue_schema_path = (
    f"{volume_root}/system/schema/autoloader_rescue_test"
)
rescue_checkpoint_path = (
    f"{volume_root}/system/checkpoints/autoloader_rescue_test"
)
rescue_table = (
    f"{catalog}.{schema}.lab03_taxi_rescue_test"
)

dbutils.fs.rm(rescue_schema_path, recurse=True)
dbutils.fs.rm(rescue_checkpoint_path, recurse=True)
spark.sql(f"DROP TABLE IF EXISTS {rescue_table}")

base_columns = [
    "pickup_datetime",
    "dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "pickup_location_id",
    "dropoff_location_id",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "total_amount",
]

explicit_schema = (
    spark.table(file_bronze_table)
    .select(*base_columns)
    .schema
)

print(f"Rescue test table: {rescue_table}")

In [0]:
from pyspark.sql.functions import col, current_timestamp

rescue_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option(
        "cloudFiles.schemaLocation",
        rescue_schema_path
    )
    .option(
        "cloudFiles.schemaEvolutionMode",
        "rescue"
    )
    .option(
        "rescuedDataColumn",
        rescued_data_column
    )
    .schema(explicit_schema)
    .load(staging_malformed_path)
    .withColumn(
        "_source_file",
        col("_metadata.file_path")
    )
    .withColumn(
        "_source_file_name",
        col("_metadata.file_name")
    )
    .withColumn(
        "_ingested_at",
        current_timestamp()
    )
)

rescue_query = (
    rescue_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        rescue_checkpoint_path
    )
    .trigger(availableNow=True)
    .toTable(rescue_table)
)

rescue_query.awaitTermination()

print("Rescued-data test completed.")

In [0]:
rescue_result_df = spark.table(rescue_table)

rescued_rows_df = (
    rescue_result_df
    .filter(f"{rescued_data_column} IS NOT NULL")
)

rescued_row_count = rescued_rows_df.count()

display(
    rescued_rows_df
    .select(
        "passenger_count",
        rescued_data_column,
        "_source_file_name"
    )
    .limit(20)
)

print(f"Rows containing rescued data: {rescued_row_count:,}")

if rescued_row_count == 0:
    raise RuntimeError(
        "No rescued data was found. Check the malformed source files."
    )

## 10. Final summary

In [0]:
summary_df = spark.createDataFrame(
    [
        ("New column", "source_system", "Added to Bronze"),
        (
            "Renamed field",
            "fare_amount → base_fare_amount",
            "Handled as separate columns"
        ),
        (
            "Malformed data",
            rescued_data_column,
            f"{rescued_row_count} rescued rows"
        ),
    ],
    ["test", "change", "result"]
)

display(summary_df)

print("Schema evolution tests completed.")
print("Next notebook: lab03_04_autoloader_monitoring")